# 01 — SkillCorner Data Contract: Targeted Event Semantics Gate

**Purpose:** Characterize the exact event categories, phase categories, EPV field coverage, and pose-archive integrity required to design H1–H3. This is not an exhaustive audit.

**Evidence standard:** observed column values are *not* official semantic definitions. Confirm meanings in the official SkillCorner specifications before freezing operational definitions.

**Output policy:** All notebook exports are written under `<repository>/outputs/`. The input dataset is never copied into this folder.

## 1. Data location and imports
Run the notebook from the repository root or a `notebooks/` subfolder. It reads the local competition-provided CSVs only. No new match data is downloaded.

In [1]:
from pathlib import Path
from collections import Counter
import sys, csv, json
from datetime import datetime, timezone

# Find repository root whether Jupyter starts at root or in notebooks/.
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "data" / "matches").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Cannot find data/matches above current working directory.")
OUTPUTS = ROOT / "outputs"
OUTPUTS.mkdir(parents=True, exist_ok=True)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Repository:", ROOT)
print("All generated files go to:", OUTPUTS)


Repository: /Users/fadihaddad/Documents/GitHub/SkillCorner_opendata_soccer
All generated files go to: /Users/fadihaddad/Documents/GitHub/SkillCorner_opendata_soccer/outputs


## 2. Focused field definitions
Only the selected fields are profiled; we do not recheck coordinates or reread tracking. Header absence and empty values are recorded separately.

In [2]:
FIELDS = (
    "event_type", "event_subtype", "start_type", "end_type",
    "pressing_chain", "pressing_chain_end_type",
    "simultaneous_defensive_engagement_same_target",
    "organised_defense", "defensive_structure", "carry", "pass_outcome",
    "game_interruption_after", "game_state",
)
NUMERIC = ("frame_start", "frame_end", "pressing_chain_length", "index_in_pressing_chain")
EPV = ("possession_epv_for_start","possession_epv_for_end","pass_epv_total")
PHASE = ("team_in_possession_phase_type", "team_out_of_possession_phase_type")
def clean(value):
    if value is None or value == "": return "<MISSING>"
    return str(value).strip()[:100]
def counter_report(c, n=25):
    return [{"value":k, "count":v} for k,v in c.most_common(n)]
def profile_csv(path, fields):
    counters = {x: Counter() for x in fields}
    coverage=Counter()
    rows=0
    with path.open(newline="", encoding="utf-8-sig") as f:
        reader=csv.DictReader(f)
        header=reader.fieldnames or []
        for row in reader:
            rows += 1
            for key in fields:
                if key in row:
                    val=clean(row.get(key))
                    counters[key][val]+=1
                    if val!="<MISSING>": coverage[key]+=1
    return {"rows":rows, "fields_present":[x for x in fields if x in header],
            "coverage_counts":dict(coverage),
            "values":{k:counter_report(v) for k,v in counters.items() if k in header}}


## 3. Collect event and phase distributions across all match CSV files
Counts are descriptive, not independent scientific observations. In later inferential models, defensive episodes and match groups—not individual frames—will define the uncertainty structure.

In [3]:
report={"generated_utc": datetime.now(timezone.utc).isoformat(), "source": "User-supplied local SkillCorner match files", "matches": {}, "pose_archives": {}}
for folder in sorted((ROOT / "data" / "matches").iterdir()):
    if not (folder.is_dir() and folder.name.isdigit()):
        continue
    mid=folder.name
    result={}
    for suffix, label, fields in (("dynamic_events.csv", "events", FIELDS + NUMERIC + EPV), ("phases_of_play.csv", "phases", PHASE)):
        f=folder / f"{mid}_{suffix}"
        result[label] = profile_csv(f, fields) if f.exists() else {"error": "MISSING"}
    report["matches"][mid] = result
print("Matches profiled:", len(report["matches"]))

Matches profiled: 20


## 4. Summarize the most important field coverage
Here, *present* means in the CSV header and *non-empty* means a row actually has a value. This guards against assuming that header presence implies usable samples.

In [4]:
key_fields = ["pressing_chain", "pressing_chain_end_type", "simultaneous_defensive_engagement_same_target", "carry", "organised_defense", "possession_epv_for_start", "pass_epv_total"]
total_matches = len(report["matches"])
for key in key_fields:
    present = sum(key in d["events"].get("fields_present", []) for d in report["matches"].values())
    populated = sum(d["events"].get("coverage_counts", {}).get(key, 0) for d in report["matches"].values())
    print(f"{key:54s} header matches={present:2d}/{total_matches} non-empty rows={populated:,}")


pressing_chain                                         header matches=20/20 non-empty rows=17,400
pressing_chain_end_type                                header matches=20/20 non-empty rows=2,382
simultaneous_defensive_engagement_same_target          header matches=20/20 non-empty rows=17,400
carry                                                  header matches=20/20 non-empty rows=18,792
organised_defense                                      header matches=20/20 non-empty rows=63,742
possession_epv_for_start                               header matches=16/20 non-empty rows=14,446
pass_epv_total                                         header matches=16/20 non-empty rows=12,035


## 5. Verify official body pose archive checksums
This uses SkillCorner's `verify_download` helper, which compares SHA-256 against their manifest. It reads compressed archives only for integrity checking and does not extract or process millions of joints.

In [5]:
try:
    from src.data.pose_loading import verify_download
except ImportError as exc:
    verify_download = None
    print("Official pose helper unavailable:", exc)
for mid in ("1925299", "1996435"):
    archive = ROOT / "pose_data" / f"{mid}.jsonl.zip"
    verified = bool(verify_download(archive)) if archive.exists() and verify_download else None
    report["pose_archives"][mid] = {"exists": archive.exists(), "bytes": archive.stat().st_size if archive.exists() else None, "verified_sha256": verified}
    print(mid, report["pose_archives"][mid])

1925299 {'exists': True, 'bytes': 623408231, 'verified_sha256': True}
1996435 {'exists': True, 'bytes': 644327811, 'verified_sha256': True}


## 6. Export lightweight hand-off results
This JSON contains frequency tables and coverage, **not raw match rows**. Keep full match files out of the GitHub submission.

In [6]:
OUTFILE = OUTPUTS / "P0_SEMANTICS_REPORT.json"
OUTFILE.write_text(json.dumps(report, indent=2), encoding="utf-8")
print("Wrote:", OUTFILE)


Wrote: /Users/fadihaddad/Documents/GitHub/SkillCorner_opendata_soccer/outputs/P0_SEMANTICS_REPORT.json


## 7. Scientific decision gate (to be completed after reviewing the report)

- [ ] Confirm category definitions using SkillCorner official Dynamic Events and Phases of Play specifications.
- [ ] Determine whether `pressing_chain` describes consecutive individual presses rather than coordinated outlet suppression.
- [ ] Determine whether `carry` marks **observed** carrying and cannot be used directly as a counterfactual candidate.
- [ ] Record pose checksum results and note any missing coverage.
- [ ] Retain EPV-missing matches for non-EPV analyses; never fill unknown EPV with zero.
- [ ] Freeze episode eligibility, predictors available at decision time, and downstream outcomes; then advance to P1.

**Stop rule:** no additional exploratory audits unless a finding threatens joins, temporal leakage, label correctness, or the hypothesis tests.